# Advanced Multi-Agent RAG Features Testing

This notebook focuses on implementing and testing advanced RAG features:
1. **Memory** (LangGraph Checkpoint)
2. **Cross-Encoder Reranking** (Using `sentence-transformers` for cost-efficiency)
3. **Semantic Chunking**
4. **Advanced Planning**

In [ ]:
import os
from dotenv import load_dotenv
from typing import List, Optional, Literal, Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import CrossEncoder
import operator

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

## 1. Memory / Conversation Persistence

We use `MemorySaver` to persist the graph state across multiple turns.

In [ ]:
class AgentState(BaseModel):
    question: str
    history: Annotated[List[BaseMessage], operator.add] = []
    context: Annotated[List[str], operator.add] = []
    answer: str = ""

def chat_node(state: AgentState):
    system_msg = SystemMessage(content="You are a helpful assistant with memory.")
    messages = state.history + [HumanMessage(content=state.question)]
    response = llm.invoke([system_msg] + messages)
    return {
        "history": [HumanMessage(content=state.question), response],
        "answer": response.content
    }

workflow = StateGraph(AgentState)
workflow.add_node("chat", chat_node)
workflow.set_entry_point("chat")
workflow.add_edge("chat", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

# Testing Memory
config = {"configurable": {"thread_id": "demo-123"}}
app.invoke({"question": "Hi, I am Khang"}, config)
res = app.invoke({"question": "What is my name?"}, config)
print("AI Response:", res['answer'])

## 2. Cross-Encoder Reranking (Standard Approach)

Instead of using an LLM (expensive), we use a Cross-Encoder like `ms-marco-MiniLM` to re-score retrieval results locally.

In [ ]:
# Load Cross-Encoder (local model, low cost)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_with_cross_encoder(query: str, documents: List[str], top_n: int = 2):
    """Re-scores and filters documents using Cross-Encoder."""
    if not documents: return []
    
    # Prepare pairs for comparison: [[query, doc1], [query, doc2], ...]
    pairs = [[query, doc] for doc in documents]
    scores = reranker.predict(pairs)
    
    # Combine docs with scores and sort
    scored_docs = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scored_docs[:top_n]]

mock_docs = [
    "Linear regression models the linear relationship between variables.",
    "A pizza is an Italian savory dish.",
    "The cat sat on the mat.",
    "Simple linear regression is a statistical method for one-variable prediction."
]

refined_docs = rerank_with_cross_encoder("What is linear regression?", mock_docs)
print("Reranked Documents:", refined_docs)

## 3. Semantic Chunking

Using semantic gaps between sentence embeddings to create chunks.

In [ ]:
semantic_splitter = SemanticChunker(embeddings)
text = "Linear regression is math. It uses lines. Sushi is fish. It uses rice. Coding is art. It uses logic."
chunks = semantic_splitter.create_documents([text])
for i, c in enumerate(chunks):
    print(f"Chunk {i+1}: {c.page_content}")